<a href="https://colab.research.google.com/github/TyrGunllod/TELECOMX_ETL/blob/main/TelecomX_BR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📌 Extracão

In [147]:
!pip install unidecode

Importação das Bibliotecas

In [148]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unidecode



Carregamento dos dados

In [149]:
dados_base = pd.read_json("https://raw.githubusercontent.com/TyrGunllod/TELECOMX_ETL/refs/heads/main/dados/TelecomX_Data.json")
dados_base.head(2)

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


#🔧 Transformação

Nova base sem as colunas aninhadas para concatenação

In [150]:
df_cust_churn = dados_base.drop(columns=['customer', 'phone', 'internet', 'account'])
df_cust_churn

,customerID,Churn
0,0002-ORFBO,No
1,0003-MKNFE,No
2,0004-TLHLJ,Yes
3,0011-IGKFF,Yes
4,0013-EXCHZ,Yes
...,...,...
7262,9987-LUTYD,No
7263,9992-RRAMN,Yes
7264,9992-UJOEL,No
7265,9993-LHIEB,No


Normaliza várias colunas aninhadas

In [151]:
df_customer = pd.json_normalize(dados_base["customer"])
df_phone    = pd.json_normalize(dados_base["phone"])
df_internet = pd.json_normalize(dados_base["internet"])
df_account  = pd.json_normalize(dados_base["account"])


Junta tudo no mesmo DataFrame

In [152]:
df_base_norm = pd.concat([df_cust_churn, df_customer, df_phone, df_internet, df_account], axis=1)
df_base_norm.head(3)

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85


Verifica a quantidade de linhas e colunas

In [153]:
df_base_norm.shape

(7267, 21)

Verifica valores nulos e o tipo

In [154]:
df_base_norm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7267 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contract          7267 non-null   object 


Verifica as possibilidades de resposta das colunas

In [163]:
for col in df_base_norm.columns:
    unicos = df_base_norm[col].unique()
    print(f"Coluna: {col}")
    print(f"Quantidade de valores únicos: {len(unicos)}")
    print(f"Valores únicos: {unicos}")
    print("-" * 50)

Coluna: customerID
Quantidade de valores únicos: 7043
Valores únicos: ['0002-ORFBO' '0003-MKNFE' '0004-TLHLJ' ... '9992-UJOEL' '9993-LHIEB'
 '9995-HOTOH']
--------------------------------------------------
Coluna: Churn
Quantidade de valores únicos: 2
Valores únicos: ['No' 'Yes']
--------------------------------------------------
Coluna: gender
Quantidade de valores únicos: 2
Valores únicos: ['Female' 'Male']
--------------------------------------------------
Coluna: SeniorCitizen
Quantidade de valores únicos: 2
Valores únicos: [0 1]
--------------------------------------------------
Coluna: Partner
Quantidade de valores únicos: 2
Valores únicos: ['Yes' 'No']
--------------------------------------------------
Coluna: Dependents
Quantidade de valores únicos: 2
Valores únicos: ['Yes' 'No']
--------------------------------------------------
Coluna: tenure
Quantidade de valores únicos: 73
Valores únicos: [ 9  4 13  3 71 63  7 65 54 72  5 56 34  1 45 50 23 55 26 69 37 49 66 67
 20 43 59 12 

Removendo registros nulos do churn

In [164]:
remover = df_base_norm.query('Churn == ""').index
df_base_norm.drop(remover, axis=0, inplace=True)
df_base_norm['Churn'].unique()

array(['No', 'Yes'], dtype=object)

Transforma todos os dados em string para normalização

In [165]:
df_base_norm = df_base_norm.astype(str)
df_base_norm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        7043 non-null   object
 1   Churn             7043 non-null   object
 2   gender            7043 non-null   object
 3   SeniorCitizen     7043 non-null   object
 4   Partner           7043 non-null   object
 5   Dependents        7043 non-null   object
 6   tenure            7043 non-null   object
 7   PhoneService      7043 non-null   object
 8   MultipleLines     7043 non-null   object
 9   InternetService   7043 non-null   object
 10  OnlineSecurity    7043 non-null   object
 11  OnlineBackup      7043 non-null   object
 12  DeviceProtection  7043 non-null   object
 13  TechSupport       7043 non-null   object
 14  StreamingTV       7043 non-null   object
 15  StreamingMovies   7043 non-null   object
 16  Contract          7043 non-null   object
 17  PaperlessBilling  7

Normaliza nomes das colunas

In [166]:
df_base_norm.columns = (
    df_base_norm.columns
    .str.strip()                # remove espaços no começo/fim
    .str.lower()                # tudo minúsculo
    .str.replace(" ", "_")      # troca espaço por "_"
    .str.replace(r"[^a-z0-9_]", "", regex=True)  # remove caracteres especiais
)

df_base_norm.columns = [unidecode.unidecode(col) for col in df_base_norm.columns]
df_base_norm.head()

,customerid,churn,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,...,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,chargesmonthly,chargestotal
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


Normaliza dados em todas as colunas object

In [167]:
for col in df_base_norm.select_dtypes(include="object").columns:
    df_base_norm[col] = df_base_norm[col].str.strip().str.lower().str.replace(" ", "_")

df_base_norm.head()

,customerid,churn,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,...,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,chargesmonthly,chargestotal
0,0002-orfbo,no,female,0,yes,yes,9,yes,no,dsl,...,yes,no,yes,yes,no,one_year,yes,mailed_check,65.6,593.3
1,0003-mknfe,no,male,0,no,no,9,yes,yes,dsl,...,no,no,no,no,yes,month-to-month,no,mailed_check,59.9,542.4
2,0004-tlhlj,yes,male,0,no,no,4,yes,no,fiber_optic,...,no,yes,no,no,no,month-to-month,yes,electronic_check,73.9,280.85
3,0011-igkff,yes,male,1,yes,no,13,yes,no,fiber_optic,...,yes,yes,no,yes,yes,month-to-month,yes,electronic_check,98.0,1237.85
4,0013-exchz,yes,female,1,yes,no,3,yes,no,fiber_optic,...,no,no,yes,yes,no,month-to-month,yes,mailed_check,83.9,267.4


In [168]:
df_base_norm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerid        7043 non-null   object
 1   churn             7043 non-null   object
 2   gender            7043 non-null   object
 3   seniorcitizen     7043 non-null   object
 4   partner           7043 non-null   object
 5   dependents        7043 non-null   object
 6   tenure            7043 non-null   object
 7   phoneservice      7043 non-null   object
 8   multiplelines     7043 non-null   object
 9   internetservice   7043 non-null   object
 10  onlinesecurity    7043 non-null   object
 11  onlinebackup      7043 non-null   object
 12  deviceprotection  7043 non-null   object
 13  techsupport       7043 non-null   object
 14  streamingtv       7043 non-null   object
 15  streamingmovies   7043 non-null   object
 16  contract          7043 non-null   object
 17  paperlessbilling  7

Alterando as colunas com valores yes/no, para true/false

In [169]:
df_base_norm = df_base_norm.replace({"Yes": True, "No": False})
df_base_norm.head()

,customerid,churn,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,...,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,chargesmonthly,chargestotal
0,0002-orfbo,no,female,0,yes,yes,9,yes,no,dsl,...,yes,no,yes,yes,no,one_year,yes,mailed_check,65.6,593.3
1,0003-mknfe,no,male,0,no,no,9,yes,yes,dsl,...,no,no,no,no,yes,month-to-month,no,mailed_check,59.9,542.4
2,0004-tlhlj,yes,male,0,no,no,4,yes,no,fiber_optic,...,no,yes,no,no,no,month-to-month,yes,electronic_check,73.9,280.85
3,0011-igkff,yes,male,1,yes,no,13,yes,no,fiber_optic,...,yes,yes,no,yes,yes,month-to-month,yes,electronic_check,98.0,1237.85
4,0013-exchz,yes,female,1,yes,no,3,yes,no,fiber_optic,...,no,no,yes,yes,no,month-to-month,yes,mailed_check,83.9,267.4


Reordena as colunas, colocando os dados booleanos para o final

In [171]:
# Lista na ordem desejada
nova_ordem = ["customerid", "gender", "tenure", "contract", "paymentmethod",
              "chargesmonthly", "chargestotal", "paperlessbilling",
              "dependents", "seniorcitizen", "partner",
              "phoneservice", "multiplelines", "internetservice",
              "onlinesecurity", "onlinebackup", "deviceprotection",
              "techsupport", "streamingtv", "streamingmovies", "churn"]

# Reorganiza o DataFrame
df_base_norm = df_base_norm[nova_ordem]

df_base_norm.head()

,customerid,gender,tenure,contract,paymentmethod,chargesmonthly,chargestotal,paperlessbilling,dependents,seniorcitizen,...,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,churn
0,0002-orfbo,female,9,one_year,mailed_check,65.6,593.3,yes,yes,0,...,yes,no,dsl,no,yes,no,yes,yes,no,no
1,0003-mknfe,male,9,month-to-month,mailed_check,59.9,542.4,no,no,0,...,yes,yes,dsl,no,no,no,no,no,yes,no
2,0004-tlhlj,male,4,month-to-month,electronic_check,73.9,280.85,yes,no,0,...,yes,no,fiber_optic,no,no,yes,no,no,no,yes
3,0011-igkff,male,13,month-to-month,electronic_check,98.0,1237.85,yes,no,1,...,yes,no,fiber_optic,no,yes,yes,no,yes,yes,yes
4,0013-exchz,female,3,month-to-month,mailed_check,83.9,267.4,yes,no,1,...,yes,no,fiber_optic,no,no,no,yes,yes,no,yes


Criar nova base, removendo os campos desnecessários

In [175]:
colunas_uteis = ["customerid", "gender", "tenure", "contract", "paymentmethod",
              "chargesmonthly", "chargestotal", "dependents", "seniorcitizen",
              "partner", "phoneservice", "multiplelines", "internetservice",
              "churn"]
df_base_util = df_base_norm[colunas_uteis]
df_base_util.head()

,customerid,gender,tenure,contract,paymentmethod,chargesmonthly,chargestotal,dependents,seniorcitizen,partner,phoneservice,multiplelines,internetservice,churn
0,0002-orfbo,female,9,one_year,mailed_check,65.6,593.3,yes,0,yes,yes,no,dsl,no
1,0003-mknfe,male,9,month-to-month,mailed_check,59.9,542.4,no,0,no,yes,yes,dsl,no
2,0004-tlhlj,male,4,month-to-month,electronic_check,73.9,280.85,no,0,no,yes,no,fiber_optic,yes
3,0011-igkff,male,13,month-to-month,electronic_check,98.0,1237.85,no,1,yes,yes,no,fiber_optic,yes
4,0013-exchz,female,3,month-to-month,mailed_check,83.9,267.4,no,1,yes,yes,no,fiber_optic,yes


In [176]:
df_base_util.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 0 to 7266
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   customerid       7043 non-null   object
 1   gender           7043 non-null   object
 2   tenure           7043 non-null   object
 3   contract         7043 non-null   object
 4   paymentmethod    7043 non-null   object
 5   chargesmonthly   7043 non-null   object
 6   chargestotal     7043 non-null   object
 7   dependents       7043 non-null   object
 8   seniorcitizen    7043 non-null   object
 9   partner          7043 non-null   object
 10  phoneservice     7043 non-null   object
 11  multiplelines    7043 non-null   object
 12  internetservice  7043 non-null   object
 13  churn            7043 non-null   object
dtypes: object(14)
memory usage: 825.4+ KB


#📊 Carga e análise

#📄Relatorio Final